# 02. Market Basket Analysis (Rekomendasi Produk)
**Proyek**: Solusi Prima Packaging Study Case  
**Peran**: Data Analyst  
**Tujuan**: Menganalisis produk yang sering dibeli bersamaan oleh pelanggan untuk menyusun strategi rekomendasi paket produk (bundling).

---

### Import Libraries & Load Cleaned Data

In [12]:
import logging
from pathlib import Path
from typing import Dict, List
import pandas as pd

# =============================================================================
# KONFIGURASI LOGGING
# Menggunakan logging standar (bukan print) agar output memiliki timestamp dan
# severity level — memudahkan debugging saat pipeline dijalankan ulang.
# getLogger(__name__) memastikan log bisa difilter per modul jika notebook ini
# diimpor sebagai bagian dari pipeline yang lebih besar.
# =============================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# =============================================================================
# KONFIGURASI PATH
# =============================================================================
DEFAULT_PATH_PROCESSED: Path = Path("..") / "data" / "processed"


# =============================================================================
# FUNGSI PEMUATAN DATA (DATA INGESTION)
# =============================================================================

def load_cleaned_datasets(filenames: List[str], input_dir: Path = DEFAULT_PATH_PROCESSED) -> Dict[str, pd.DataFrame]:
    """Memuat sekumpulan file CSV hasil pembersihan dari direktori penyimpanan.

    Fungsi ini memeriksa keberadaan file di sistem lokal sebelum melakukan pembacaan
    untuk menghindari kegagalan eksekusi di tengah jalan.

    Args:
        filenames (List[str]): Daftar nama file CSV yang ingin dimuat (tanpa path).
        input_dir (Path, optional): Jalur direktori data bersih berbasis pathlib.Path.
            Nilai default diarahkan ke DEFAULT_PATH_PROCESSED.

    Returns:
        Dict[str, pd.DataFrame]: Kamus berisi nama dasar file (stem) sebagai key
            dan objek DataFrame yang berhasil dimuat sebagai value.

    Raises:
        FileNotFoundError: Jika salah satu file yang dikonfigurasi tidak ditemukan di disk.
    """
    loaded_data: Dict[str, pd.DataFrame] = {}
    
    for filename in filenames:
        filepath = input_dir / filename
        
        if not filepath.exists():
            raise FileNotFoundError(f"File data bersih tidak ditemukan pada jalur: '{filepath}'")
            
        label = filepath.stem.replace("cleaned_data_", "")
        logger.info("Memuat dataset bersih [%s] dari: %s", label, filepath)
        loaded_data[label] = pd.read_csv(filepath)
        
    return loaded_data


# =============================================================================
# FUNGSI PENGGABUNGAN DATA (DATA MERGING)
# =============================================================================

def merge_transaction_with_products(df_transaksi: pd.DataFrame, df_produk: pd.DataFrame) -> pd.DataFrame:
    """Menggabungkan data transaksi dengan atribut produk berbasis left join.

    Sebelum melakukan merge, fungsi ini memvalidasi keberadaan kunci relasi (foreign key)
    dan kolom dimensi target untuk mencegah penurunan kualitas data akibat kolom kosong.

    Args:
        df_transaksi (pd.DataFrame): DataFrame log transaksi utama.
        df_produk (pd.DataFrame): DataFrame master data produk.

    Returns:
        pd.DataFrame: DataFrame hasil penggabungan yang kaya akan atribut dimensi produk.

    Raises:
        KeyError: Jika kunci relasi atau kolom deskriptif produk tidak ditemukan di dataset.
    """
    # Kolom wajib untuk proses penggabungan
    join_key = "product_id"
    required_product_cols = ["product_id", "nama_produk", "kategori"]
    
    # Validasi fail-fast kolom transaksi
    if join_key not in df_transaksi.columns:
        raise KeyError(f"Kunci relasi '{join_key}' tidak ditemukan pada DataFrame transaksi.")
        
    # Validasi fail-fast kolom master produk
    missing_cols = [col for col in required_product_cols if col not in df_produk.columns]
    if missing_cols:
        raise KeyError(f"Kolom wajib master produk berikut tidak ditemukan: {missing_cols}")

    logger.info("Menggabungkan data transaksi dengan atribut deskriptif master produk.")
    
    df_merged = df_transaksi.merge(
        df_produk[required_product_cols],
        on=join_key,
        how="left"
    )
    
    return df_merged


# =============================================================================
# EKSEKUSI PIPELINE ANALISIS (CELL NOTEBOOK)
# =============================================================================

# 1. Definisikan target file yang akan dibaca
target_files = ["cleaned_data_transaksi.csv", "cleaned_data_produk.csv"]

# 2. Eksekusi pemuatan data secara kolektif
datasets = load_cleaned_datasets(filenames=target_files)

# 3. Eksekusi penggabungan data analitis
df_analytical = merge_transaction_with_products(
    df_transaksi=datasets["transaksi"],
    df_produk=datasets["produk"]
)

logger.info("Data berhasil dimuat dan digabungkan. Siap untuk dianalisis.")

2026-06-23 07:40:37 | INFO     | __main__ | Memuat dataset bersih [transaksi] dari: ..\data\processed\cleaned_data_transaksi.csv
2026-06-23 07:40:37 | INFO     | __main__ | Memuat dataset bersih [produk] dari: ..\data\processed\cleaned_data_produk.csv
2026-06-23 07:40:37 | INFO     | __main__ | Menggabungkan data transaksi dengan atribut deskriptif master produk.
2026-06-23 07:40:37 | INFO     | __main__ | Data berhasil dimuat dan digabungkan. Siap untuk dianalisis.


### Strategi: Memeriksa Pola Transaksi
Kita perlu memeriksa apakah produk dibeli bersamaan dalam satu nomor order (order_id) atau kita harus melihat kecenderungan pembelian di tingkat pelanggan (customer_id).


In [14]:
# =============================================================================
# FUNGSI ANALISIS PERILAKU TRANSAKSI
# =============================================================================

def analyze_order_product_diversity(df: pd.DataFrame) -> pd.Series:
    """Menghitung keberagaman jenis produk unik per order_id.

    Fungsi ini memvalidasi keberadaan kolom yang dibutuhkan sebelum melakukan
    proses agregasi untuk memastikan integritas eksekusi.

    Args:
        df (pd.DataFrame): DataFrame hasil penggabungan (merged) yang berisi 
            data transaksi dan produk.

    Returns:
        pd.Series: Seri data dengan order_id sebagai indeks dan jumlah produk unik 
            sebagai nilai variabelnya.

    Raises:
        KeyError: Jika kolom 'order_id' atau 'product_id' tidak ditemukan.
    """
    required_cols = ["order_id", "product_id"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        raise KeyError(f"Gagal melakukan analisis. Kolom berikut tidak ditemukan: {missing_cols}")

    logger.info("Menghitung jumlah ragam jenis produk unik per order ID.")
    return df.groupby("order_id")["product_id"].nunique()


def log_transaction_habits_summary(orders_habit: pd.Series) -> None:
    """Mencetak ringkasan statistik keranjang belanja ke sistem logging.

    Menampilkan total transaksi keseluruhan dan volume transaksi yang memiliki 
    sifat multi-produk (keranjang belanja berisi lebih dari 1 jenis produk).

    Args:
        orders_habit (pd.Series): Seri data hasil dari fungsi `analyze_order_product_diversity`.
    """
    total_orders = len(orders_habit)
    
    # Menyimpan kondisi evaluasi ke dalam variabel boolean untuk efisiensi komputasi
    is_multi_product = orders_habit > 1
    total_multi_product_orders = int(is_multi_product.sum())
    
    # Menghitung persentase multi-product order sebagai metrik tambahan produksi
    pct_multi_product = (total_multi_product_orders / total_orders * 100) if total_orders > 0 else 0.0

    logger.info("=== ANALISIS PERILAKU BELANJA (SHOPPING HABIT) ===")
    logger.info("Total Transaksi (Order): {:,}".format(total_orders))
    logger.info("Jumlah transaksi dengan > 1 jenis produk (Multi-Prod): {:,} ({:.2f}%)".format(
        total_multi_product_orders, pct_multi_product
    ))


# =============================================================================
# EKSEKUSI (CELL NOTEBOOK)
# =============================================================================

# 1. Hitung metrik keberagaman produk per transaksi
orders_habit = analyze_order_product_diversity(df_analytical)

# 2. Cetak ringkasan analisis perilaku ke log
log_transaction_habits_summary(orders_habit)

2026-06-23 07:43:35 | INFO     | __main__ | Menghitung jumlah ragam jenis produk unik per order ID.
2026-06-23 07:43:35 | INFO     | __main__ | === ANALISIS PERILAKU BELANJA (SHOPPING HABIT) ===
2026-06-23 07:43:35 | INFO     | __main__ | Total Transaksi (Order): 100
2026-06-23 07:43:35 | INFO     | __main__ | Jumlah transaksi dengan > 1 jenis produk (Multi-Prod): 0 (0.00%)


**Catatan:** Seluruh order_id berisi 1 produk saja, maka asosiasi produk akan dihitung berdasarkan kesamaan 'customer_id' (riwayat belanja pelanggan).

### Membuat Matriks Pasangan Produk / Co-occurrence Matrix

In [24]:
"""Module Analisis Asosiasi Produk (Market Basket Analysis).

Modul ini menyediakan fungsi untuk menghitung frekuensi kombinasi dua produk
yang dibeli bersamaan oleh pelanggan yang sama, serta menyediakan utilitas
untuk menampilkan hasil akhir dengan format visual yang rapi di Jupyter Notebook.
"""

from typing import Any
from IPython.display import display

# =============================================================================
# INITIALIZATION & CONFIGURATION
# =============================================================================

logger = logging.getLogger(__name__)

DEFAULT_OUTPUT_DIR: Path = Path("..") / "outputs" / "tables"


# =============================================================================
# VISUAL RENDERING CONFIGURATION (CSS)
# =============================================================================

TABLE_BASE_CSS: List[Dict[str, Any]] = [
    {
        "selector": "thead th",
        "props": [
            ("background-color", "#F3F5F7"),
            ("color",            "#1558B0"),
            ("font-family",      "monospace"),
            ("font-size",        "12px"),
            ("font-weight",      "700"),
            ("border-bottom",    "2px solid #1558B0"),
            ("padding",          "8px 12px"),
            ("text-align",       "left !important"),
        ]
    },
    {
        "selector": "tbody td",
        "props": [
            ("background-color", "#FFFFFF"),
            ("color",            "#0B1D35"),
            ("font-family",      "monospace"),
            ("font-size",        "12px"),
            ("border-bottom",    "1px solid #D8DCE1"),
            ("padding",          "6px 12px"),
            ("text-align",       "left !important"),
        ]
    },
    {
        "selector": "table",
        "props": [
            ("border-collapse", "collapse"),
            ("border",          "1px solid #D8DCE1"),
            ("background",      "#FFFFFF"),
            ("margin",          "0"),
        ]
    },
    {
        "selector": "td.col2",
        "props": [
            ("text-align",       "center !important")
        ]
    }
]


# =============================================================================
# CORE BUSINESS LOGIC FUNCTIONS
# =============================================================================

def calculate_product_associations(df: pd.DataFrame) -> pd.DataFrame:
    """Menghitung frekuensi kombinasi pasangan produk berbasis perilaku pelanggan.

    Fungsi ini mengisolasi ID pelanggan dan produk untuk mengeksekusi self-merge
    secara efisien, mengeliminasi pasangan produk identik, menghindari
    perhitungan ganda (duplikasi permutasi terbalik), dan mengurutkan hasil
    berdasarkan frekuensi tertinggi.

    Args:
        df (pd.DataFrame): DataFrame gabungan utama yang minimal mengandung
            kolom 'customer_id', 'product_id', dan 'nama_produk'.

    Returns:
        pd.DataFrame: DataFrame berisi kolom 'nama_produk_A', 'nama_produk_B',
            dan 'jumlah_transaksi' yang diurutkan secara descending.

    Raises:
        KeyError: Jika satu atau lebih kolom wajib tidak ditemukan di DataFrame.
    """
    required_cols = ["customer_id", "product_id", "nama_produk"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        raise KeyError(f"Gagal memproses asosiasi. Kolom wajib tidak ditemukan: {missing_cols}")

    logger.info("Memulai ekstraksi kombinasi produk berbasis transaksi pelanggan.")
    
    df_slim = df[required_cols].drop_duplicates()
    df_pairs = df_slim.merge(df_slim, on="customer_id", suffixes=("_A", "_B"))
    
    valid_pairs_mask = df_pairs["product_id_A"] < df_pairs["product_id_B"]
    df_pairs_filtered = df_pairs.loc[valid_pairs_mask]
    
    logger.info("Menghitung matriks frekuensi kemunculan bersamaan (co-occurrence).")
    associations = (
        df_pairs_filtered.groupby(["nama_produk_A", "nama_produk_B"])
        .size()
        .reset_index(name="jumlah_transaksi")
    )
    
    associations = associations.sort_values(by="jumlah_transaksi", ascending=False).reset_index(drop=True)
    return associations


def export_associations(df_associations: pd.DataFrame, output_dir: Path = DEFAULT_OUTPUT_DIR) -> None:
    """Menyimpan DataFrame hasil analisis asosiasi ke file lokal dalam format CSV.

    Fungsi ini menjamin keandalan proses I/O dengan mengonstruksi direktori
    tujuan secara otomatis jika belum tersedia di dalam penyimpanan lokal.

    Args:
        df_associations (pd.DataFrame): Dataframe hasil perhitungan asosiasi produk.
        output_dir (Path, optional): Jalur direktori penyimpanan berbasis Path.
            Nilai default mengarah ke DEFAULT_OUTPUT_DIR.

    Raises:
        OSError: Jika sistem operasi gagal membuat direktori atau menulis file ke disk.
    """
    filepath = output_dir / "top_product_pairs.csv"
    
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        df_associations.to_csv(filepath, index=False)
        logger.info("Tabel asosiasi berhasil disimpan secara aman di: '%s'", filepath)
    except OSError as exc:
        logger.error("Gagal mengekspor data asosiasi ke jalur '%s'. Detail: %s", filepath, exc)
        raise


# =============================================================================
# PIPELINE EXECUTION (NOTEBOOK CELL)
# =============================================================================

logger.info("=== MENGHITUNG KOMBINASI PRODUK YANG DIBELI BERSAMAAN ===")

product_associations = calculate_product_associations(df_merged)

logger.info("Top 5 Pasangan Produk Teratas:")
styled_top_5 = (
    product_associations.head(5)
    .style
    .set_table_styles(TABLE_BASE_CSS)
    .hide(axis="index")
)

display(styled_top_5)

export_associations(product_associations)

2026-06-23 07:56:19 | INFO     | __main__ | === MENGHITUNG KOMBINASI PRODUK YANG DIBELI BERSAMAAN ===
2026-06-23 07:56:19 | INFO     | __main__ | Memulai ekstraksi kombinasi produk berbasis transaksi pelanggan.
2026-06-23 07:56:19 | INFO     | __main__ | Menghitung matriks frekuensi kemunculan bersamaan (co-occurrence).
2026-06-23 07:56:19 | INFO     | __main__ | Top 5 Pasangan Produk Teratas:


nama_produk_A,nama_produk_B,jumlah_transaksi
Botol PET Bening 500ml,Standing Pouch Aluminium 250g,9
Botol PET Bening 500ml,Botol Kaca Amber 250ml,8
Botol PET Bening 500ml,Tray Makanan Mika 4 Sekat,7
Botol PET Bening 500ml,Stiker Label Vinyl Tahan Air,7
Botol PET Bening 500ml,Tutup Botol Ulir 28mm,7


2026-06-23 07:56:19 | INFO     | __main__ | Tabel asosiasi berhasil disimpan secara aman di: '..\outputs\tables\top_product_pairs.csv'


### Visualisasi Cross-Sell Produk

In [51]:
import plotly.graph_objects as go
import plotly.io as pio

# Jalur ekspor relatif dari direktori runtime notebook
DEFAULT_CHART_DIR: Path = Path("..") / "outputs" / "charts"

# Palet warna korporat standar McKinsey untuk menjaga hierarki visual acuan eksekutif
COLORS: Dict[str, str] = {
    "navy": "#0B1D35",       # Warna utama teks dan aksen penting
    "tick": "#6B7B8D",       # Warna label sumbu (axis ticks)
    "footnote": "#9DA8B5",   # Warna teks catatan kaki (informasi sekunder)
    "primary": "#1558B0",    # Warna batang dominan (peringkat 1)
    "secondary": "#8FB4E8",  # Warna batang sekunder (peringkat 2)
    "muted": "#DCE6F4",      # Warna latar untuk data non-kontekstual (peringkat 3-5)
    "grid": "rgba(11,29,53,0.05)",
    "bg": "#FFFFFF"
}

# Kamus pemetaan untuk lokalisasi nama produk ke laporan resmi Bahasa Indonesia
RENAME_MAP: Dict[str, str] = {
    "Botol PET Bening 500ml": "Botol PET",
    "Standing Pouch Aluminium 250g": "Stand Pouch",
    "Botol Kaca Amber 250ml": "Botol Kaca Amber",
    "Tutup Botol Ulir 28mm": "Tutup Botol",
    "Stiker Label Vinyl Tahan Air": "Stiker Label",
    "Tray Makanan Mika 4 Sekat": "Mika Makanan",
    "Dus Produk Glossy 10x10x10": "Dus Glossy",
    "Karton Box Die Cut 20x15x8": "Karton Box"
}

# Mengambil top 5 kombinasi terkuat untuk visualisasi eksekutif yang fokus
chart_df = product_associations.sort_values("jumlah_transaksi", ascending=False).head(5).copy()

# Menggunakan .get() untuk fallback aman jika ada produk baru yang belum terdaftar di RENAME_MAP
chart_df["A"] = chart_df["nama_produk_A"].apply(lambda x: RENAME_MAP.get(x, x))
chart_df["B"] = chart_df["nama_produk_B"].apply(lambda x: RENAME_MAP.get(x, x))
chart_df["pair"] = chart_df["A"] + " + " + chart_df["B"]

# Perhitungan insight dinamis berdasarkan komposisi produk jangkar (Anchor Product)
total_rows = len(chart_df)
pet_count = chart_df["pair"].str.contains("Botol PET").sum()
# Defensif terhadap ZeroDivisionError jika dataframe input kosong
share = round(pet_count / total_rows * 100) if total_rows > 0 else 0

title_text = "Botol PET Mendorong Peluang Cross-Selling Terbesar"
subtitle_text = f"Botol PET muncul dalam {share}% dari kombinasi pembelian terkuat, menjadikannya produk jangkar paling efektif untuk strategi promosi berbasis bundling."

# Pewarnaan bertingkat (Muted Colorway) untuk mengarahkan fokus audiens langsung ke peringkat teratas
bar_colors = [
    COLORS["primary"],
    COLORS["secondary"],
    COLORS["muted"],
    COLORS["muted"],
    COLORS["muted"]
]

# Penentuan batas sumbu X secara dinamis (+15% buffer ruang kosong) agar label nilai tidak terpotong
max_val = chart_df["jumlah_transaksi"].max() if not chart_df.empty else 10
x_max_range = max_val * 1.15

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=chart_df["jumlah_transaksi"],
        y=chart_df["pair"],
        orientation="h",
        marker=dict(color=bar_colors),
        textfont=dict(size=12, color=COLORS["navy"]),
        hovertemplate="<b>%{y}</b><br>Dibeli Bersamaan: %{x} kali<extra></extra>"
    )
)

# Konfigurasi kanvas mengikuti layout laporan formal dengan margin t&b longgar untuk judul multi-baris
fig.update_layout(
    paper_bgcolor=COLORS["bg"],
    plot_bgcolor=COLORS["bg"],
    width=1120,
    height=630,
    margin=dict(l=60, r=40, t=80, b=80),
    title=dict(
        text=(
            f"<b>{title_text}</b>"
            "<br>"
            "<span style='font-size:13px; color:#6B7B8D; font-weight:normal;'>"
            f"{subtitle_text}"
            "</span>"
        ),
        x=0.07,
        y=0.94,
        font=dict(family="Arial", size=20, color=COLORS["navy"])
    ),
    font=dict(family="Arial", color=COLORS["navy"]),
    showlegend=False
)

# Urutan dibalik (autorange='reversed') agar urutan peringkat 1 tetap berada di posisi paling atas chart
fig.update_yaxes(
    autorange="reversed",
    title="",
    tickfont=dict(family="Arial", size=13, color=COLORS["navy"]),
    tickangle=0,
    ticklabelstandoff=10
)

# Menyembunyikan grid lines untuk mempertahankan aspek visual minimalis khas McKinsey Chart
fig.update_xaxes(
    title=dict(
        text="Frekuensi Pembelian Bersamaan", 
        font=dict(family="Arial", size=13, color=COLORS["navy"]),
        standoff=15
    ),
    showgrid=False,
    zeroline=False,
    tickfont=dict(family="Arial", size=11, color=COLORS["tick"]),
    range=[0, x_max_range]
)

# Catatan kaki diletakkan secara relatif menggunakan koordinat kertas (paper reference)
fig.add_annotation(
    x=0,
    y=-0.22,
    xref="paper",
    yref="paper",
    showarrow=False,
    align="left",
    text="Sumber: Market Basket Analysis. Lima kombinasi produk teratas diurutkan berdasarkan frekuensi pembelian bersamaan.",
    font=dict(family="Arial", size=10, color=COLORS["footnote"])
)

# Manajemen penulisan berkas dengan penanganan error sistem lokal (I/O)
try:
    DEFAULT_CHART_DIR.mkdir(parents=True, exist_ok=True)
    output_path = DEFAULT_CHART_DIR / "cross_sell_opportunity_analysis.png"
    # Skala ditingkatkan (scale=3) untuk memastikan resolusi gambar tajam saat disematkan di dokumen/PPT
    pio.write_image(fig, str(output_path), scale=3)
except OSError as exc:
    logger.error("Sistem gagal menulis gambar chart ke disk. Detail: %s", exc)
    raise

fig.show()
print(f"Chart saved to: {output_path}")

2026-06-23 13:32:01 | INFO     | choreographer.browsers.chromium | Chromium init'ed with kwargs {}
2026-06-23 13:32:01 | INFO     | choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
2026-06-23 13:32:11 | INFO     | choreographer.utils._tmpfile | shutil.rmtree worked.
2026-06-23 13:32:11 | INFO     | choreographer.utils._tmpfile | Temp directory created: C:\Users\Vertin\AppData\Local\Temp\tmpnqhdgihl.
2026-06-23 13:32:11 | INFO     | choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
2026-06-23 13:32:21 | INFO     | choreographer.utils._tmpfile | shutil.rmtree worked.
2026-06-23 13:32:21 | INFO     | choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
2026-06-23 13:32:21 | INFO     | choreographer.browser_async | Opening browser.
2026-06-23 13:32:21 | INFO     | choreographer.utils._tmpfile | shutil.rmtree worked.
2026-06-23 13:32:31 | INFO     | choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
2026-06-23 13:32:31 | I

Chart saved to: ..\outputs\charts\cross_sell_opportunity_analysis.png


### 1. Temuan Utama (*Key Insights*)

* **Botol PET sebagai Produk Jangkar (*Anchor Product*)**: Botol PET muncul dalam 100% dari lima kombinasi pembelian terkuat. Hal ini menunjukkan bahwa Botol PET merupakan produk dengan daya tarik tertinggi yang memicu transaksi produk kemasan lainnya.
* **Kombinasi Terkuat**: Pasangan produk **Botol PET + Stand Pouch** menempati urutan pertama dengan frekuensi pembelian bersama tertinggi (mencapai angka 9 pada grafik), diikuti oleh **Botol PET + Botol Kaca Amber** di posisi kedua (angka 8).
* **Kombinasi Pendukung**: Kombinasi Botol PET dengan **Mika Makanan**, **Stiker Label**, dan **Tutup Botol** memiliki tingkat kekuatan yang sama (berada di angka 7). Keterkaitan dengan Stiker Label dan Tutup Botol menunjukkan pola perilaku konsumen yang ingin membeli kebutuhan pengemasan secara fungsional dan siap pakai dalam satu kali transaksi.

### 2. Implikasi Strategis untuk *Reseller* dan *Marketplace*

Melihat pola asosiasi produk yang sangat kuat ini, beberapa strategi praktis berbasis data yang dapat diterapkan adalah:

* **Strategi Promosi Paket Bundling**:
* **Paket Utama**: Buat paket promo "Bundling UMKM Minuman & Camilan" yang menggabungkan Botol PET dengan Stand Pouch dengan diskon khusus. Strategi ini sangat relevan karena kedua produk ini paling sering dicari bersamaan oleh pelaku usaha.
* **Paket Fungsional**: Buat produk komplementer otomatis, seperti paket "Botol PET + Tutup + Stiker Label" sebagai satu kesatuan produk siap pakai untuk mempermudah konsumen baru.


* **Optimalisasi Penempatan Produk di Platform (*Cross-Selling* Otomatis)**:
* Manfaatkan algoritma AI untuk memunculkan rekomendasi otomatis di keranjang belanja (*add-to-cart*). Ketika konsumen memasukkan Botol PET, sistem harus langsung merekomendasikan Stand Pouch atau Botol Kaca Amber sebelum mereka melakukan *checkout*.


* **Manajemen Stok Terintegrasi (*Inventory Alignment*)**:
* *Reseller* harus menjaga sinkronisasi stok. Jika stok Botol PET sedang melimpah, *reseller* wajib memastikan persediaan Stand Pouch, Tutup Botol, dan Stiker Label juga aman. Kekosongan pada produk pelengkap akan langsung menurunkan potensi penjualan profitabel dari Botol PET itu sendiri.